# NextAI — Qwen3.5-0.8B engineering student training

This notebook trains the single compact NextAI student. It downloads bounded CAD/BIM teaching samples, performs vision-and-language BF16/FP16 LoRA SFT, evaluates held-out data, and exports a Q4_K_M model with a compact Q8_0 vision projector. Teacher weights, raw datasets and training libraries are never copied into the Next3D installer.

**Runtime:** choose a GPU runtime. The first pass trains one full epoch over the prepared student corpus. A basic 0.8B LoRA run can fit on a small GPU, but multi-view CAD examples, 4K context and image batches may need more headroom. The notebook checks available CUDA and VRAM before proceeding.

Swiss Dwellings and GNI IFC corpora are optional, large, training-only downloads. They stay in the cloud training workspace and never enter the 1.2 GB Next3D package. Actual native CAD/BIM application validation is still required before promoting a model.

In [ ]:
import os, subprocess, sys, json, torch
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then rerun.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
REPO = '/content/Next3D'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1','--branch','nextai-operator-foundation','https://github.com/28royalpapi/Next3D.git',REPO],check=True)
os.chdir(REPO)
print('Training branch:', subprocess.check_output(['git','branch','--show-current'],text=True).strip())

In [ ]:
%pip install --upgrade --quiet unsloth unsloth_zoo 'transformers>=5,<6' trl datasets peft accelerate bitsandbytes pillow shapely


## 1. Prepare the teaching material

The Autodesk Zero-to-CAD 100K set supplies multi-view CAD examples and construction-operation traces. The IFC/BIM instruction and QA sets supply BIM/IFC concepts. Next3D-authored lessons fill 2D drafting, parametric sketches, documentation and construction workflow gaps. Only bounded samples are downloaded.

### Optional: real architectural floor plans and authored IFC models

Set either flag to `True` to add a larger training source. These downloads are not part of the Next3D installer. Swiss Dwellings supplies 2D WKT geometry and room labels; GNI supplies authored IFC2x3/IFC4 models. The GNI corpus is student coursework, so the parsed content teaches IFC structure—not engineering correctness. IFC4-generated 73K models are documented as a later optional extension.


In [ ]:
ENABLE_SWISS_DWELLINGS = False
ENABLE_GNI_IFC_MODELS = False
SWISS_CORPUS = '/content/swiss-dwellings-corpus'
IFC_CORPUS = '/content/gni-ifc-corpus'

if ENABLE_SWISS_DWELLINGS:
    swiss_zip = '/content/swiss-dwellings-v3.0.0.zip'
    if not os.path.isfile(swiss_zip):
        subprocess.run(['wget','-c','-O',swiss_zip,'https://zenodo.org/records/7788422/files/swiss-dwellings-v3.0.0.zip?download=1'],check=True)
    subprocess.run([
        sys.executable, 'training/nextai/prepare_swiss_dwellings.py',
        '--zip', swiss_zip, '--output-dir', SWISS_CORPUS,
        '--max-train', '600', '--max-validation', '60', '--max-test', '60',
    ],check=True)

if ENABLE_GNI_IFC_MODELS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'ifcopenshell'],check=True)
    gni_root = '/content/gni-ifc-source'
    os.makedirs(gni_root, exist_ok=True)
    downloads = [
        ('2025_BIMfundamentals.zip','https://zenodo.org/records/19722012/files/2025_BIMfundamentals.zip?download=1'),
        ('2026_BIMprojects.zip','https://zenodo.org/records/19722012/files/2026_BIMprojects.zip?download=1'),
    ]
    import zipfile
    for filename, url in downloads:
        archive_path = os.path.join('/content', filename)
        if not os.path.isfile(archive_path):
            subprocess.run(['wget','-c','-O',archive_path,url],check=True)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(gni_root)
    subprocess.run([
        sys.executable, 'training/nextai/prepare_ifc_models.py',
        '--root', gni_root, '--output-dir', IFC_CORPUS,
        '--source-name', 'GNI-BIM Dataset', '--doi', '10.5281/zenodo.19722012',
        '--license', 'CC-BY-4.0', '--max-train', '160',
        '--max-validation', '24', '--max-test', '24',
    ],check=True)


In [ ]:
DATA_DIR = '/content/nextai-corpus'
NEXTCAD_NATIVE_FILES = {
    'train': None,
    'validation': None,
    'test': None,
}
NEXTBIM_NATIVE_FILES = {
    'train': None,
    'validation': None,
    'test': None,
}

# Build the pinned native capability census before training.
# Upload nextcad-runtime.json from the native FreeCAD build artifact to /content/
# to teach from every live command and discovered Python API member.
CENSUS_DIR = '/content/nextai-capability-census'
NEXTCAD_RUNTIME_REPORT = '/content/nextcad-runtime.json'
os.makedirs(CENSUS_DIR, exist_ok=True)
census_args = [
    sys.executable, 'training/nextai/capability_census.py',
    '--bim-catalog', 'docs/next-ai/catalogs/BIMCRAFT_NATIVE_COMMANDS.v1.json',
    '--nextcad-source-index', 'docs/next-ai/catalogs/NEXTCAD_WORKBENCH_SOURCE_INDEX.v1.json',
    '--expected-nextcad-commit', '00cb280454069966a2fb0a8e7dc184c46e7b38b3',
    '--expected-nextbim-commit', 'a0c57b738e308d433ab4b8c0c93d7fff9e53bc1b',
    '--output-dir', CENSUS_DIR,
]
if os.path.isfile(NEXTCAD_RUNTIME_REPORT):
    census_args += ['--nextcad-runtime-report', NEXTCAD_RUNTIME_REPORT]
else:
    print('WARNING: nextcad-runtime.json is not present. The census will contain source-indexed modules and Next BIM commands, but not the complete live NextCAD command/API inventory.')
subprocess.run(census_args, check=True)
CAPABILITY_CENSUS = CENSUS_DIR + '/capability-census.json'
with open(CENSUS_DIR + '/summary.json') as f:
    capability_summary = json.load(f)
print('Pinned native capabilities:', json.dumps(capability_summary, indent=2))

# Optional: add the real NextCAD native acceptance trace from the CI artifact.
# It is held out as test data and is never moved into train/validation.
ACCEPTANCE_TRACE = '/content/nextcad-ai-native-acceptance-trace.jsonl'
if os.path.isfile(ACCEPTANCE_TRACE):
    ACCEPTANCE_SFT = '/content/nextcad-acceptance-sft'
    subprocess.run([
        sys.executable, 'training/nextai/prepare_native_sft.py',
        '--input', ACCEPTANCE_TRACE,
        '--output-dir', ACCEPTANCE_SFT,
    ], check=True)
    candidate_test = ACCEPTANCE_SFT + '/test.jsonl'
    if os.path.isfile(candidate_test) and os.path.getsize(candidate_test):
        NEXTCAD_NATIVE_FILES['test'] = candidate_test

builder_args = [
    sys.executable, 'training/nextai/prepare_public_corpora.py',
    '--output-dir', DATA_DIR,
    '--max-cad-train', '2500', '--max-cad-validation', '250', '--max-cad-test', '250',
    '--max-bim-train', '30000', '--max-bim-validation', '3000',
    '--max-ifcqa-train', '5000', '--max-ifcqa-test', '1000',
    '--max-mixed-train', '24000', '--max-validation', '2500', '--max-test', '1200',
    '--capability-census', CAPABILITY_CENSUS,
]
if ENABLE_SWISS_DWELLINGS:
    builder_args += ['--swiss-dwellings-dir', SWISS_CORPUS]
if ENABLE_GNI_IFC_MODELS:
    builder_args += ['--ifc-model-corpus-dir', IFC_CORPUS]
for split, key in [('train','train'), ('validation','validation'), ('test','test')]:
    if NEXTCAD_NATIVE_FILES[key]:
        builder_args += [f'--nextcad-native-{split}', NEXTCAD_NATIVE_FILES[key]]
    if NEXTBIM_NATIVE_FILES[key]:
        builder_args += [f'--nextbim-native-{split}', NEXTBIM_NATIVE_FILES[key]]
subprocess.run(builder_args,check=True)
with open(DATA_DIR + '/manifest.json') as f: manifest=json.load(f)
print(json.dumps({
    k: manifest.get(k) for k in [
        'student','train_records','validation_records','test_records',
        'training_domain_counts','source_counts'
    ]
},indent=2))


The base is the post-trained Qwen3.5-0.8B checkpoint so the student retains its general instruction-following behaviour. We add a small LoRA adapter to both the vision and language pathways. We do not use QLoRA for this Qwen3.5 recipe. This matters: the student must learn from CAD drawings and multi-view 3D geometry, not only from text. The original checkpoint weights remain intact; only the small adapter is trained.


In [ ]:
OUTPUT_DIR = '/content/nextai-run-001'
RESUME_CHECKPOINT = None  # Set to a saved checkpoints/checkpoint-N directory to resume.
train_command = [
    sys.executable, 'training/nextai/train_nextai.py',
    '--data-dir', DATA_DIR,
    '--output-dir', OUTPUT_DIR,
    '--base-model', 'Qwen/Qwen3.5-0.8B',
    '--max-steps', '-1',
    '--num-train-epochs', '1',
    '--max-seq-length', '4096',
    '--batch-size', '1',
    '--gradient-accumulation', '8',
    '--lora-rank', '16',
    '--save-merged-model',
]
if RESUME_CHECKPOINT:
    train_command += ['--resume-from-checkpoint', RESUME_CHECKPOINT]
subprocess.run(train_command, check=True)


## 2B. Compact the vision projector

Qwen3.5 is multimodal. The language-model GGUF is not the whole model: a separate vision projector is required for image input. This step converts the projector to Q8_0, verifies the GGUF header and checksum, and removes the larger full-precision projector only after the replacement passes validation.

In [ ]:
LLAMA_CPP = '/content/llama.cpp'
if not os.path.isdir(LLAMA_CPP):
    subprocess.run(['git','clone','--depth','1','https://github.com/ggml-org/llama.cpp.git',LLAMA_CPP],check=True)
subprocess.run([
    sys.executable, 'training/nextai/compact_multimodal_export.py',
    '--merged-model-dir', OUTPUT_DIR + '/merged_16bit',
    '--llama-cpp-dir', LLAMA_CPP,
    '--gguf-dir', OUTPUT_DIR + '/gguf_q4_k_m',
],check=True)
with open(OUTPUT_DIR + '/gguf_q4_k_m/compact-export-manifest.json') as f: compact=json.load(f)
print('Compact model bytes:', compact['total_gguf_bytes'])
print(json.dumps(compact['files'],indent=2))

## 3. Compare the trained student against the untouched model

These are language/structure diagnostics only. The final acceptance test must execute proposed actions in the actual FreeCAD and BIMCraft engines and validate the resulting native projects.

In [ ]:
# Evaluate the base and tuned model on the complete, identical held-out test set.
subprocess.run([
    sys.executable, 'training/nextai/evaluate_nextai.py',
    '--model', 'Qwen/Qwen3.5-0.8B',
    '--test-jsonl', DATA_DIR + '/test.jsonl',
    '--output', OUTPUT_DIR + '/evaluation-base.json',
],check=True)
subprocess.run([
    sys.executable, 'training/nextai/evaluate_nextai.py',
    '--model', OUTPUT_DIR + '/merged_16bit',
    '--test-jsonl', DATA_DIR + '/test.jsonl',
    '--output', OUTPUT_DIR + '/evaluation-tuned.json',
],check=True)
for name in ['evaluation-base.json','evaluation-tuned.json']:
    with open(OUTPUT_DIR + '/' + name) as f:
        report=json.load(f)
    print('\n',name,'test records:',report['test_records'])
    print(json.dumps(report['metrics_by_domain'],indent=2))


## 4. Check the compact export size

The exported model is only a candidate. Do not replace the packaged model unless it passes the held-out benchmark and native CAD/BIM acceptance suite. This notebook reports the exact model-file sizes so the full Next3D release budget can be measured separately.

In [ ]:
result_path = OUTPUT_DIR + '/training_result.json'
with open(result_path) as f: result=json.load(f)
print('GGUF total bytes:', result.get('gguf_total_bytes', 'not exported'))
for item in result.get('gguf_files', []): print(item)
print('Candidate output:', OUTPUT_DIR)

## 5. Persist the training candidate for individual laptops\n\nThe Colab runtime is temporary. This step copies only the inference candidate, adapter, configuration and evaluation/provenance reports to Google Drive. It does not copy the raw training corpus or training cache. A candidate is not promoted until local multimodal runtime and native CAD/BIM acceptance pass.

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
PERSISTENT_DIR = '/content/drive/MyDrive/Next3D/NextAI/nextai-run-001'
os.makedirs(PERSISTENT_DIR, exist_ok=True)

for name in [
    'training_result.json',
    'training_config.json',
    'evaluation-base.json',
    'evaluation-tuned.json',
]:
    source_path = os.path.join(OUTPUT_DIR, name)
    if os.path.isfile(source_path):
        shutil.copy2(source_path, PERSISTENT_DIR)

for folder in ['adapter', 'gguf_q4_k_m']:
    source_path = os.path.join(OUTPUT_DIR, folder)
    if os.path.isdir(source_path):
        shutil.copytree(source_path, os.path.join(PERSISTENT_DIR, folder), dirs_exist_ok=True)

shutil.copy2(DATA_DIR + '/manifest.json', os.path.join(PERSISTENT_DIR, 'training-data-manifest.json'))
shutil.copy2(CENSUS_DIR + '/capability-census.json', os.path.join(PERSISTENT_DIR, 'capability-census.json'))
shutil.copy2(CENSUS_DIR + '/summary.json', os.path.join(PERSISTENT_DIR, 'capability-census-summary.json'))

candidate_files = []
for root, _, names in os.walk(PERSISTENT_DIR):
    for name in names:
        full_path = os.path.join(root, name)
        candidate_files.append({
            'path': os.path.relpath(full_path, PERSISTENT_DIR),
            'size_bytes': os.path.getsize(full_path),
        })
candidate_manifest = {
    'base_model': 'Qwen/Qwen3.5-0.8B',
    'candidate_only': True,
    'native_cad_bim_promotion': 'pending independent acceptance',
    'vision_runtime_acceptance': 'pending local runtime smoke test',
    'files': candidate_files,
    'total_bytes': sum(item['size_bytes'] for item in candidate_files),
}
with open(os.path.join(PERSISTENT_DIR, 'candidate-package-manifest.json'), 'w') as f:
    json.dump(candidate_manifest, f, indent=2)
print('Candidate persisted to:', PERSISTENT_DIR)
print(json.dumps(candidate_manifest, indent=2))
